# CO2 Methanation in a Catalytic Plate Reactor Modeled as a Chain of Well Stirred Reactors 

In [1]:
import csv
import math

import cantera as ct
ct.CanteraError.set_stack_trace_depth(10)

print(ct.__version__)

# unit conversion factors to SI
cm = 0.01 # cm to m
mm = 0.001 # mm to m
mg = 0.001 # mg to g
ml = 1e-6 # ml to m^3
minute = 1 / 60.0 # min to s
bar = 1e5

4.0.0a1


# Input Parameters

In [2]:

tc = 320 # Temperatuer in Celcius
t = tc + 273  # Temperature in Kelvin

p = 1.2 * bar # pressure

length_total = 100 * mm  # Reactor length
length_coated = 55 * mm # Coated length

width = 40 * mm # Reactor width
height = 5 * mm # reactor height

area = width * height # cross sectional area

coated_volume = length_coated * width * height # Volume of the coated section in the reactor

ni_specific_area = 23.1 # specific surface area of ni in m^2 / g catalyst

cat_mass = 83.3 * mg # mass of the catalyst 

cat_area = ni_specific_area * cat_mass # surface area of the catalyst

cat_area_per_vol = cat_area / coated_volume # Catalyst particle surface area per unit volume

volumetric_flow_rate = 100 * ml * minute  # volumetric flow rate

#To start with, I will assume a porisity of 1
porosity = 1  # Catalyst bed porosity

# input file containing the surface reaction mechanism
yaml_file = '/home/austin/Desktop/Kreitz Lab Work/methanation-model/data/File_0_chem_covdep_edited_automatically.yaml'

output_filename = '/home/austin/Desktop/Kreitz Lab Work/methanation-model/results/CO2_methanation_model_output.csv'


In [3]:
# The PFR will be simulated by a chain of 'NReactors' stirred reactors.
NReactors = 201

# import the phase models and set the initial conditions
surf = ct.Interface(yaml_file, name='surface1')
surf.TP = t, p
gas = surf.adjacent['gas']
gas.TPX = t, ct.one_atm, 'Ar:20, H2(4):64, CO2(2):16'

rlen = length_total/(NReactors-1)
rvol = area * rlen * porosity

# catalyst area in one reactor
cat_area = cat_area_per_vol * rvol

mass_flow_rate = volumetric_flow_rate * gas.density

# The plug flow reactor is represented by a linear chain of zero-dimensional
# reactors. The gas at the inlet to the first one has the specified inlet
# composition, and for all others the inlet composition is fixed at the
# composition of the reactor immediately upstream. Since in a PFR model there
# is no diffusion, the upstream reactors are not affected by any downstream
# reactors, and therefore the problem may be solved by simply marching from
# the first to last reactor, integrating each one to steady state.

cov = surf.coverages

# Create the reactor network

In [4]:
# create a new reactor
r = ct.IdealGasReactor(gas, energy='off')
r.volume = rvol

# create a reservoir to represent the reactor immediately upstream. Note
# that the gas object is set already to the state of the upstream reactor
upstream = ct.Reservoir(gas, name='upstream')

# create a reservoir for the reactor to exhaust into. The composition of
# this reservoir is irrelevant.
downstream = ct.Reservoir(gas, name='downstream')

# Add the reacting surface to the reactor. The area is set to the desired
# catalyst area in the reactor.
rsurf = ct.ReactorSurface(surf, r, A=cat_area)

# The mass flow rate into the reactor will be fixed by using a
# MassFlowController object.
m = ct.MassFlowController(upstream, r, mdot=mass_flow_rate)

# We need an outlet to the downstream reservoir. This will determine the
# pressure in the reactor. The value of K will only affect the transient
# pressure difference.
v = ct.PressureController(r, downstream, primary=m, K=1e-6)

sim = ct.ReactorNet([r])

CanteraError: 
*******************************************************************************
CanteraError thrown by getElementWeight:
element not found: T
-------------------------------------------------------------------------------
 0# Cantera::CanteraError::CanteraError(std::__cxx11::basic_string<char, std::char_traits<char>, std::allocator<char> > const&) in /home/austin/miniforge3/envs/ct-4.0/lib/libcantera_shared.so.4
 1# Cantera::CanteraError::CanteraError<>(std::__cxx11::basic_string<char, std::char_traits<char>, std::allocator<char> > const&, std::__cxx11::basic_string<char, std::char_traits<char>, std::allocator<char> > const&) in /home/austin/miniforge3/envs/ct-4.0/lib/python3.13/site-packages/cantera/_cantera.cpython-313-x86_64-linux-gnu.so
 2# 0x00007475557B2A49 in /home/austin/miniforge3/envs/ct-4.0/lib/libcantera_shared.so.4
 3# Cantera::Phase::addElement(std::__cxx11::basic_string<char, std::char_traits<char>, std::allocator<char> > const&, double, int, double, int) 

# Run the Simulation

In [ ]:
output_data = []

for n in range(NReactors):
    
    dist = n * rlen * 1.0e3  # distance in mm

    # only run in the simulation if over the coated area

    if dist >= 35 and dist <= 90:

        # Set the state of the reservoir to match that of the previous reactor
        upstream.phase.TDY = r.phase.TDY
        sim.reinitialize()
        
        sim.advance_to_steady_state()

    # write the gas mole fractions and surface coverages vs. distance
    
    output_data.append(
        [dist, r.T - 273.15, r.phase.P / ct.one_atm]
        + list(r.phase.X)  # use r.phase.X not gas.X
        + list(rsurf.phase.coverages)  # use rsurf.phase.coverages not surf.coverages
    )

with open(output_filename, 'w', newline="") as outfile:
    writer = csv.writer(outfile)
    writer.writerow(['Distance (mm)', 'T (C)', 'P (atm)'] +
                    gas.species_names + surf.species_names)
    writer.writerows(output_data)

